In [2]:
#import sys
#!{sys.executable} -m pip install duckduckgo-search langchain-experimental

# Built-in Tools 

In [9]:
from langchain_community.tools import DuckDuckGoSearchRun
search_run = DuckDuckGoSearchRun()
result = search_run.invoke("Is there a 2% charge on UPI transactions above ₹2,000, and who actually pays the applicable MDR?")
print(result)

UPI Payment Charges Rules: Ordinary UPI users will not pay transaction fees for payments above Rs 2,000. Person-to-person UPI transactions will remain completely free, regardless of the amount transferred. Payments to merchants up to Rs 2,000 will also continue to be free of charge. A zero-MDR framework will protect eligible small merchants from the new charges. These updated provisions will ... The latest changes around UPI charges relate to the Merchant Discount Rate (MDR) applicable to certain merchant transactions, while person-to-person (P2P) transfers will continue to remain free. 📰 UPI Payments New Rule Explained: What Happens to Payments Above INR 2,000 From October 15. Will UPI transfers over Rs 2,000 attract fees? Get clarity on MDR, free P2P payments, and what the latest government notification means for users and merchants. Government proposes changes to the Payment and Settlement Systems Act. This could allow a Merchant Discount Rate on select UPI transactions. Payments ab

In [10]:
from langchain_community.tools import ShellTool
shell_tool = ShellTool()
result = shell_tool.invoke("whoami")
print(result)

Executing command:
 whoami
aayushaswal



/Users/aayushaswal/genai-agentic-lab/venv/lib/python3.14/site-packages/langchain_community/tools/shell/tool.py:33: UserWarning: The shell tool has no safeguards by default. Use at your own risk.
  warnings.warn(


# Custom-Tools


In [12]:
# Method 1 — @tool decorator

from langchain_core.tools import tool

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

In [13]:
result = multiply.invoke({"a": 3, "b": 5})
print(result)

15


In [14]:
print(multiply.name)
print(multiply.description)
print(multiply.args)

multiply
Multiply two numbers.
{'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}


In [15]:
print(multiply.args_schema.model_json_schema())

{'description': 'Multiply two numbers.', 'properties': {'a': {'title': 'A', 'type': 'integer'}, 'b': {'title': 'B', 'type': 'integer'}}, 'required': ['a', 'b'], 'title': 'multiply', 'type': 'object'}


In [20]:
# Method 2 — StructuredTool + Pydantic

from langchain_core.tools import StructuredTool
from pydantic import BaseModel, Field

class MultiplyInput(BaseModel):
    a: int = Field(..., description="The first number to multiply")
    b: int = Field(..., description="The second number to multiply")

def multiply_func(a: int, b: int) -> int:
    return a * b

multiply_tool = StructuredTool.from_function(
    func=multiply_func,
    name="multiply",
    description="Multiply two numbers",
    args_schema=MultiplyInput,
)

In [21]:
result = multiply_tool.invoke({"a": 3, "b": 3})
print(result)
print(multiply_tool.name)
print(multiply_tool.description)
print(multiply_tool.args)

9
multiply
Multiply two numbers
{'a': {'description': 'The first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to multiply', 'title': 'B', 'type': 'integer'}}


In [25]:
# Method 3 — BaseTool subclass

from langchain_core.tools import BaseTool
from pydantic import BaseModel, Field
from typing import Type

# input schema (same Pydantic model as before)
class MultiplyInput(BaseModel):
    a: int = Field(..., description="The first number to multiply")
    b: int = Field(..., description="The second number to multiply")

class MultiplyTool(BaseTool):
    name: str = "multiply"
    description: str = "Multiply two numbers"
    args_schema: Type[BaseModel] = MultiplyInput

    def _run(self, a: int, b: int) -> int:
        return a * b

In [26]:
multiply_tool = MultiplyTool()

result = multiply_tool.invoke({"a": 3, "b": 3})
print(result)   
print(multiply_tool.name, multiply_tool.description, multiply_tool.args)

9
multiply Multiply two numbers {'a': {'description': 'The first number to multiply', 'title': 'A', 'type': 'integer'}, 'b': {'description': 'The second number to multiply', 'title': 'B', 'type': 'integer'}}


# Toolkits

In [27]:
from langchain_core.tools import tool

@tool
def add(a: int, b: int) -> int:
    """Add two numbers."""
    return a + b

@tool
def multiply(a: int, b: int) -> int:
    """Multiply two numbers."""
    return a * b

class MathToolkit:
    def get_tools(self):
        return [add, multiply]

In [28]:
toolkit = MathToolkit()
tools = toolkit.get_tools()

for t in tools:
    print(t.name, "=>", t.description)

add => Add two numbers.
multiply => Multiply two numbers.
